In [ ]:
import sys
import os
from pathlib import Path
import requests
import time
import json

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

# Find project root and add to Python path
current_dir = Path.cwd()
if current_dir.name == "module6" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = Path("/mnt/data/Portfolio/RAG")

if project_root.exists():
    print(f"Project root: {project_root}")
    sys.path.insert(0, str(project_root))
else:
    print("Project root not found - check directory structure")

print("Environment setup complete")

# ──────────────────────────────────────────────────────
# Check Service Health Including Week 6 Services
print("\nWEEK 6 SERVICE HEALTH CHECK")
print("=" * 40)

services = {
    "FastAPI": "http://localhost:8000/api/v1/health",
    "OpenSearch": "http://localhost:9200/_cluster/health",
    "Ollama": "http://localhost:11434/api/version",
    "LangFuse": "http://localhost:3000/api/public/health"
}

all_healthy = True
for service_name, url in services.items():
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            print(f"✓ {service_name}: Healthy")
        else:
            print(f"✗ {service_name}: HTTP {response.status_code}")
            all_healthy = False
    except Exception as e:
        print(f"✗ {service_name}: Not accessible - {e}")
        all_healthy = False

# Check Redis separately via redis-py
print("\nChecking Redis:")
try:
    import redis
    r = redis.Redis(host="localhost", port=6379, socket_connect_timeout=3)
    r.ping()
    print("✓ Redis: Healthy")
except Exception as e:
    print(f"✗ Redis: Not accessible - {e}")
    all_healthy = False

print()
if all_healthy:
    print("✓ All services ready for Week 6!")
else:
    print("⚠ Some services need attention. Run: docker compose up --build -d")